# Build Monthly Panel DataFrames — Strictly Historical Features**POINT-IN-TIME STRICT**: All features use data **STRICTLY BEFORE** the target month.| Suffix | Meaning ||---|---|| `_hist_` | Cumulative from all months before target month || `_prev_` | Value from the immediately preceding month (lag 1) || `_prev` | Active snapshot as of end of previous month || Vitals/Labs/Meds/ADL/GG | Monthly stats lagged by 1 month |**NO same-month (`_curr_`) features are included.**

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
DATA_DIR = r'C:\Users\whita\Documents\case-tricura\case\data'
OUTPUT_DIR = r'C:\Users\whita\Documents\case-tricura\output_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(DATA_DIR)

tables = {}
for f in os.listdir('.'):
    if f.endswith('.parquet'):
        name = f.replace('.parquet', '')
        tables[name] = pd.read_parquet(f)
        print(f"  {name}: {tables[name].shape}")
print(f"\nLoaded {len(tables)} tables.")

## 2. Build Monthly Panel of Active Residents

In [ ]:
res = tables['residents'].copy()

all_dates = []
for name, df in tables.items():
    for col in df.columns:
        if df[col].dtype == 'datetime64[ns]':
            valid = df[col].dropna()
            if len(valid) > 0:
                all_dates.extend([valid.min(), valid.max()])

data_min = pd.Timestamp(min(all_dates)).normalize().replace(day=1)
data_max = pd.Timestamp(max(all_dates)).normalize().replace(day=1)
months = pd.date_range(start=data_min, end=data_max, freq='MS')
print(f"Date range: {data_min.strftime('%Y-%m')} to {data_max.strftime('%Y-%m')} ({len(months)} months)")

rows = []
for month_start in months:
    month_end = month_start + pd.offsets.MonthEnd(0)
    active = res[
        (res['admission_date'] <= month_end) &
        (res['discharge_date'].isna() | (res['discharge_date'] >= month_start))
    ].copy()
    active['year_month'] = month_start
    rows.append(active[['resident_id', 'facility_id', 'year_month',
                         'date_of_birth', 'admission_date', 'discharge_date',
                         'deceased_date', 'outpatient']])

panel = pd.concat(rows, ignore_index=True)
print(f"Panel: {len(panel):,} rows ({panel['resident_id'].nunique():,} residents)")

## 3. Resident Features (known at month start)

In [ ]:
panel['age_years'] = (panel['year_month'] - panel['date_of_birth']).dt.days / 365.25
panel['los_days'] = (panel['year_month'] - panel['admission_date']).dt.days
panel['is_outpatient'] = panel['outpatient'].astype(int)
panel.drop(columns=['date_of_birth', 'admission_date', 'discharge_date',
                     'deceased_date', 'outpatient'], inplace=True)
panel.head()

## 4. Helper Functions — Strictly Historical

In [ ]:
def merge_historical_features(panel_df, source_df, date_col, resident_col, agg_dict, prefix):
    df = source_df.copy()
    df['_date'] = pd.to_datetime(df[date_col])
    df['_ym'] = df['_date'].dt.to_period('M').dt.to_timestamp()
    monthly = df.groupby([resident_col, '_ym']).agg(**agg_dict).reset_index()
    monthly.rename(columns={'_ym': 'year_month'}, inplace=True)
    monthly = monthly.sort_values([resident_col, 'year_month'])
    for col in agg_dict.keys():
        monthly[f'{prefix}_hist_{col}'] = (
            monthly.groupby(resident_col)[col]
            .transform(lambda x: x.cumsum().shift(1, fill_value=0))
        )
        monthly[f'{prefix}_prev_{col}'] = (
            monthly.groupby(resident_col)[col].shift(1, fill_value=0)
        )
    keep_cols = [resident_col, 'year_month'] + \
        [f'{prefix}_hist_{c}' for c in agg_dict.keys()] + \
        [f'{prefix}_prev_{c}' for c in agg_dict.keys()]
    monthly = monthly[keep_cols]
    panel_df = panel_df.merge(monthly, on=[resident_col, 'year_month'], how='left')
    return panel_df

def merge_lagged_monthly_stats(panel_df, stats_df, resident_col='resident_id'):
    lagged = stats_df.copy()
    lagged['year_month'] = lagged['year_month'] + pd.DateOffset(months=1)
    feat_cols = [c for c in lagged.columns if c not in [resident_col, 'year_month']]
    lagged.rename(columns={c: f'{c}_prev' for c in feat_cols}, inplace=True)
    panel_df = panel_df.merge(lagged, on=[resident_col, 'year_month'], how='left')
    return panel_df

def compute_active_prev_month(source_df, start_col, end_col, resident_col, agg_dict, months_range):
    rows = []
    for month_start in months_range:
        prev_start = month_start - pd.DateOffset(months=1)
        prev_end = prev_start + pd.offsets.MonthEnd(0)
        active = source_df[
            (source_df[start_col] <= prev_end) &
            (source_df[end_col].isna() | (source_df[end_col] >= prev_start))
        ]
        if len(active) > 0:
            agg_result = active.groupby(resident_col).agg(**agg_dict).reset_index()
            agg_result['year_month'] = month_start
            rows.append(agg_result)
    if rows:
        return pd.concat(rows, ignore_index=True)
    else:
        return pd.DataFrame(columns=[resident_col, 'year_month'] + list(agg_dict.keys()))

print("Helpers defined.")

## 5. Feature Engineering — All Strictly Historical### 5a–c. Incidents, Injuries, Factors

In [ ]:
inc = tables['incidents'][tables['incidents']['strikeout'] == False].copy()
panel = merge_historical_features(panel, inc, 'occurred_at', 'resident_id',
    agg_dict={'count': ('incident_id', 'nunique')}, prefix='incidents')
for itype in ['Fall', 'Wound', 'Elopement', 'Altercation', 'Medication Error', 'Choking']:
    inc_sub = inc[inc['incident_type'] == itype].copy()
    safe_name = itype.lower().replace(' ', '_')
    panel = merge_historical_features(panel, inc_sub, 'occurred_at', 'resident_id',
        agg_dict={'count': ('incident_id', 'nunique')}, prefix=f'inc_{safe_name}')

inj = tables['injuries'].merge(
    inc[['incident_id', 'resident_id', 'occurred_at']].drop_duplicates(), on='incident_id', how='left')
panel = merge_historical_features(panel, inj, 'occurred_at', 'resident_id',
    agg_dict={'count': ('injury_id', 'nunique'), 'post_incident': ('is_post_incident', 'sum')}, prefix='injuries')

fac = tables['factors'].merge(
    inc[['incident_id', 'resident_id', 'occurred_at']].drop_duplicates(), on='incident_id', how='left')
panel = merge_historical_features(panel, fac.dropna(subset=['resident_id']), 'occurred_at', 'resident_id',
    agg_dict={'count': ('factor_id', 'nunique'), 'types': ('factor_type', 'nunique')}, prefix='factors')
print(f"After incidents/injuries/factors: {panel.shape}")

### 5d. Diagnoses (historical onsets + active as of prev month + ICD pivot)

In [ ]:
diag = tables['diagnoses'][tables['diagnoses']['strikeout'] == False].copy()
panel = merge_historical_features(panel, diag, 'onset_at', 'resident_id',
    agg_dict={'count': ('diagnosis_id', 'nunique'), 'distinct_icd': ('icd_10_code', 'nunique')}, prefix='diag')

diag_active = compute_active_prev_month(diag, 'onset_at', 'resolved_at', 'resident_id',
    agg_dict={'diag_active_prev_count': ('diagnosis_id', 'nunique'),
              'diag_active_prev_distinct_icd': ('icd_10_code', 'nunique')}, months_range=months)
panel = panel.merge(diag_active, on=['resident_id', 'year_month'], how='left')

diag['icd_chapter'] = diag['icd_10_code'].str[0].fillna('X')
icd_rows = []
for month_start in months:
    prev_start = month_start - pd.DateOffset(months=1)
    prev_end = prev_start + pd.offsets.MonthEnd(0)
    active_d = diag[(diag['onset_at'] <= prev_end) & (diag['resolved_at'].isna() | (diag['resolved_at'] >= prev_start))]
    if len(active_d) > 0:
        pivot = active_d.groupby(['resident_id', 'icd_chapter'])['diagnosis_id'].nunique().unstack(fill_value=0)
        pivot.columns = [f'icd_ch_{c}_prev' for c in pivot.columns]
        pivot = pivot.reset_index(); pivot['year_month'] = month_start; icd_rows.append(pivot)
if icd_rows:
    panel = panel.merge(pd.concat(icd_rows, ignore_index=True), on=['resident_id', 'year_month'], how='left')
print(f"After diagnoses: {panel.shape}")

### 5e–f. Care Plans & Needs (active as of prev month)

In [ ]:
cp = tables['care_plans'].copy()
cp_active = compute_active_prev_month(cp[cp['strikeout']==False], 'initiated_at', 'closed_at', 'resident_id',
    agg_dict={'care_plan_active_prev_count': ('care_plan_id', 'nunique')}, months_range=months)
panel = panel.merge(cp_active, on=['resident_id', 'year_month'], how='left')
panel = merge_historical_features(panel, cp, 'initiated_at', 'resident_id',
    agg_dict={'count': ('care_plan_id', 'nunique'), 'strikeout': ('strikeout', 'sum')}, prefix='careplan')

needs_clean = tables['needs'][tables['needs']['strikeout']==False]
needs_active = compute_active_prev_month(needs_clean, 'initiated_at', 'resolved_at', 'resident_id',
    agg_dict={'needs_active_prev_count': ('need_id', 'nunique'),
              'needs_active_prev_types': ('need_type', 'nunique'),
              'needs_active_prev_categories': ('need_category', 'nunique')}, months_range=months)
panel = panel.merge(needs_active, on=['resident_id', 'year_month'], how='left')

need_cat_rows = []
for month_start in months:
    prev_start = month_start - pd.DateOffset(months=1)
    prev_end = prev_start + pd.offsets.MonthEnd(0)
    active_n = needs_clean[(needs_clean['initiated_at'] <= prev_end) & (needs_clean['resolved_at'].isna() | (needs_clean['resolved_at'] >= prev_start))]
    if len(active_n) > 0:
        pivot = active_n.groupby(['resident_id', 'need_category'])['need_id'].nunique().unstack(fill_value=0)
        pivot.columns = [f'need_cat_{c.lower().replace(" ","_").replace("-","_").replace("/","_")}_prev' for c in pivot.columns]
        pivot = pivot.reset_index(); pivot['year_month'] = month_start; need_cat_rows.append(pivot)
if need_cat_rows:
    panel = panel.merge(pd.concat(need_cat_rows, ignore_index=True), on=['resident_id', 'year_month'], how='left')
print(f"After care plans & needs: {panel.shape}")

### 5g. Vitals (lagged by 1 month)

In [ ]:
vit = tables['vitals'][tables['vitals']['strikeout']==False].copy()
vit['_ym'] = vit['measured_at'].dt.to_period('M').dt.to_timestamp()
for vtype in vit['vital_type'].dropna().unique():
    vt = vit[vit['vital_type']==vtype]
    safe_vt = vtype.lower().replace(' ','_').replace('-','_')
    ms = vt.groupby(['resident_id','_ym']).agg(mean=('value','mean'),std=('value','std'),
        min_val=('value','min'),max_val=('value','max'),count=('value','count')).reset_index()
    ms.rename(columns={'_ym':'year_month'}, inplace=True)
    ms.columns = ['resident_id','year_month'] + [f'vital_{safe_vt}_{c}' for c in ['mean','std','min','max','count']]
    panel = merge_lagged_monthly_stats(panel, ms)

bp = vit[vit['dystolic_value'].notna()]
if len(bp) > 0:
    bp_m = bp.groupby(['resident_id','_ym'])['dystolic_value'].agg(['mean','std','min','max']).reset_index()
    bp_m.rename(columns={'_ym':'year_month'}, inplace=True)
    bp_m.columns = ['resident_id','year_month','bp_diastolic_mean','bp_diastolic_std','bp_diastolic_min','bp_diastolic_max']
    panel = merge_lagged_monthly_stats(panel, bp_m)
print(f"After vitals: {panel.shape}")

### 5h–i. Labs & Medications (lagged + historical)

In [ ]:
labs = tables['lab_reports'].copy()
labs['_ym'] = labs['reported_at'].dt.to_period('M').dt.to_timestamp()
lab_m = labs.groupby(['resident_id','_ym']).agg(
    lab_count=('lab_report_id','nunique'), lab_distinct_names=('lab_name','nunique'),
    lab_abnormal=('severity_status', lambda x: x.str.lower().str.contains('abnormal|critical|high|low',na=False).sum())
).reset_index().rename(columns={'_ym':'year_month'})
panel = merge_lagged_monthly_stats(panel, lab_m)

lab_sev = labs.groupby(['resident_id','_ym','severity_status'])['lab_report_id'].nunique().reset_index()
lab_sev.rename(columns={'_ym':'year_month','lab_report_id':'count'}, inplace=True)
if len(lab_sev) > 0:
    lsp = lab_sev.pivot_table(index=['resident_id','year_month'],columns='severity_status',values='count',fill_value=0)
    lsp.columns = [f'lab_sev_{str(c).lower().replace(" ","_")}' for c in lsp.columns]
    panel = merge_lagged_monthly_stats(panel, lsp.reset_index())
panel = merge_historical_features(panel, labs, 'reported_at', 'resident_id',
    agg_dict={'count': ('lab_report_id','nunique')}, prefix='labs')

med = tables['medications'].copy()
med['_ym'] = med['scheduled_at'].dt.to_period('M').dt.to_timestamp()
med_m = med.groupby(['resident_id','_ym']).agg(
    med_total=('medication_id','nunique'), med_distinct=('description','nunique')
).reset_index().rename(columns={'_ym':'year_month'})
panel = merge_lagged_monthly_stats(panel, med_m)

ms = med.groupby(['resident_id','_ym','status'])['medication_id'].nunique().reset_index()
ms.rename(columns={'_ym':'year_month','medication_id':'count'}, inplace=True)
if len(ms) > 0:
    msp = ms.pivot_table(index=['resident_id','year_month'],columns='status',values='count',fill_value=0)
    msp.columns = [f'med_status_{str(c).lower().replace(" ","_").replace("-","_")}' for c in msp.columns]
    panel = merge_lagged_monthly_stats(panel, msp.reset_index())
panel = merge_historical_features(panel, med, 'scheduled_at', 'resident_id',
    agg_dict={'count': ('medication_id','nunique')}, prefix='meds')
print(f"After labs & meds: {panel.shape}")

### 5j–k. Physician Orders & Therapy (active as of prev month)

In [ ]:
po = tables['physician_orders'].copy()
po_active = compute_active_prev_month(po, 'start_at', 'end_at', 'resident_id',
    agg_dict={'orders_active_prev_count': ('order_id','nunique'),
              'orders_active_prev_categories': ('category','nunique')}, months_range=months)
panel = panel.merge(po_active, on=['resident_id','year_month'], how='left')

po_cat_rows = []
for month_start in months:
    ps = month_start - pd.DateOffset(months=1); pe = ps + pd.offsets.MonthEnd(0)
    apo = po[(po['start_at']<=pe)&(po['end_at'].isna()|(po['end_at']>=ps))]
    if len(apo)>0:
        p = apo.groupby(['resident_id','category'])['order_id'].nunique().unstack(fill_value=0)
        p.columns = [f'order_cat_{c.lower().replace(" ","_").replace("-","_").replace("/","_")}_prev' for c in p.columns]
        p = p.reset_index(); p['year_month']=month_start; po_cat_rows.append(p)
if po_cat_rows:
    panel = panel.merge(pd.concat(po_cat_rows, ignore_index=True), on=['resident_id','year_month'], how='left')

tt = tables['therapy_tracks'].copy()
tt_active = compute_active_prev_month(tt, 'start_at', 'end_at', 'resident_id',
    agg_dict={'therapy_active_prev_count': ('therapy_id','nunique'),
              'therapy_active_prev_disciplines': ('discipline','nunique')}, months_range=months)
panel = panel.merge(tt_active, on=['resident_id','year_month'], how='left')

tt_disc_rows = []
for month_start in months:
    ps = month_start - pd.DateOffset(months=1); pe = ps + pd.offsets.MonthEnd(0)
    att = tt[(tt['start_at']<=pe)&(tt['end_at'].isna()|(tt['end_at']>=ps))]
    if len(att)>0:
        p = att.groupby(['resident_id','discipline'])['therapy_id'].nunique().unstack(fill_value=0)
        p.columns = [f'therapy_{c.lower().replace(" ","_").replace("-","_")}_prev' for c in p.columns]
        p = p.reset_index(); p['year_month']=month_start; tt_disc_rows.append(p)
if tt_disc_rows:
    panel = panel.merge(pd.concat(tt_disc_rows, ignore_index=True), on=['resident_id','year_month'], how='left')
print(f"After orders & therapy: {panel.shape}")

### 5l–p. Hospital, ADL, GG, Document Tags

In [ ]:
# Hospital transfers & admissions (historical only)
ht = tables['hospital_transfers'].copy()
panel = merge_historical_features(panel, ht, 'effective_date', 'resident_id',
    agg_dict={'count': ('transfer_id','nunique'), 'emergency': ('emergency_flag', lambda x: (x==1).sum())}, prefix='transfers')
ha = tables['hospital_admissions'].copy()
panel = merge_historical_features(panel, ha, 'effective_date', 'resident_id',
    agg_dict={'count': ('admission_id','nunique')}, prefix='hosp_adm')

# ADL (lagged)
adl = tables['adl_responses'].copy()
adl['response_num'] = pd.to_numeric(adl['response'], errors='coerce')
adl['_ym'] = adl['assessment_date'].dt.to_period('M').dt.to_timestamp()
adl_m = adl.groupby(['resident_id','_ym']).agg(
    adl_count=('adl_response_id','nunique'), adl_mean_response=('response_num','mean'),
    adl_mean_change=('adl_change','mean'),
    adl_neg_changes=('adl_change', lambda x: (x<0).sum()),
    adl_pos_changes=('adl_change', lambda x: (x>0).sum()),
    adl_distinct_activities=('activity','nunique')
).reset_index().rename(columns={'_ym':'year_month'})
panel = merge_lagged_monthly_stats(panel, adl_m)

adl_cat = adl.groupby(['resident_id','_ym','category'])['response_num'].mean().reset_index()
adl_cat.rename(columns={'_ym':'year_month'}, inplace=True)
if len(adl_cat) > 0:
    acp = adl_cat.pivot_table(index=['resident_id','year_month'],columns='category',values='response_num',fill_value=0)
    acp.columns = [f'adl_avg_{c.lower().replace(" ","_").replace("-","_")}' for c in acp.columns]
    panel = merge_lagged_monthly_stats(panel, acp.reset_index())

# GG (lagged)
gg = tables['gg_responses'].copy()
gg['_ym'] = gg['created_at'].dt.to_period('M').dt.to_timestamp()
gg_m = gg.groupby(['resident_id','_ym']).agg(
    gg_count=('gg_response_id','nunique'), gg_mean_response_code=('response_code','mean'),
    gg_mean_change=('change','mean'),
    gg_neg_changes=('change', lambda x: (x<0).sum()),
    gg_pos_changes=('change', lambda x: (x>0).sum()),
    gg_distinct_tasks=('task_name','nunique')
).reset_index().rename(columns={'_ym':'year_month'})
panel = merge_lagged_monthly_stats(panel, gg_m)

gg_grp = gg.groupby(['resident_id','_ym','task_group'])['response_code'].mean().reset_index()
gg_grp.rename(columns={'_ym':'year_month'}, inplace=True)
if len(gg_grp) > 0:
    ggp = gg_grp.pivot_table(index=['resident_id','year_month'],columns='task_group',values='response_code',fill_value=0)
    ggp.columns = [f'gg_avg_{c.lower().replace(" ","_").replace("-","_").replace("/","_")}' for c in ggp.columns]
    panel = merge_lagged_monthly_stats(panel, ggp.reset_index())

# Document tags (historical only)
dt_df = tables['document_tags'].copy()
panel = merge_historical_features(panel, dt_df, 'created_at', 'resident_id',
    agg_dict={'count': ('document_tag_id','nunique'), 'types': ('doc_type','nunique'),
              'avg_confidence': ('match_confidence','mean')}, prefix='doctags')
print(f"Final panel: {panel.shape}")

## 6. Build Targets & Export

In [ ]:
inc['_ym'] = inc['occurred_at'].dt.to_period('M').dt.to_timestamp()
CLAIM_TYPES = {
    'fall': 'Fall', 'wound': 'Wound', 'medication_error': 'Medication Error',
    'elopement': 'Elopement', 'altercation': 'Altercation', 'choking': 'Choking',
}
ht_target = tables['hospital_transfers'].copy()
ht_target['_ym'] = ht_target['effective_date'].dt.to_period('M').dt.to_timestamp()
rth_monthly = ht_target.groupby(['resident_id','_ym']).agg(
    rth_count=('transfer_id','nunique')).reset_index().rename(columns={'_ym':'year_month'})
rth_monthly['target_rth'] = 1
CLAIM_TYPES['rth'] = '__rth__'

numeric_cols = panel.select_dtypes(include=[np.number]).columns
panel[numeric_cols] = panel[numeric_cols].fillna(0)

for claim_key, claim_value in CLAIM_TYPES.items():
    df = panel.copy()
    if claim_key == 'rth':
        df = df.merge(rth_monthly[['resident_id','year_month','target_rth']], on=['resident_id','year_month'], how='left')
        df['target_rth'] = df['target_rth'].fillna(0).astype(int)
        target_col = 'target_rth'
    else:
        inc_type = inc[inc['incident_type']==claim_value]
        tm = inc_type.groupby(['resident_id','_ym']).agg(target_count=('incident_id','nunique')).reset_index().rename(columns={'_ym':'year_month'})
        tm[f'target_{claim_key}'] = 1
        df = df.merge(tm[['resident_id','year_month',f'target_{claim_key}']], on=['resident_id','year_month'], how='left')
        df[f'target_{claim_key}'] = df[f'target_{claim_key}'].fillna(0).astype(int)
        target_col = f'target_{claim_key}'
    df = df.sort_values(['resident_id','year_month']).reset_index(drop=True)
    total = len(df); positive = df[target_col].sum()
    print(f"{claim_key.upper():>20s}  {df.shape[0]:>8,} rows x {df.shape[1]:>4} cols  |  "
          f"target: {int(positive):>5,} / {total:>8,} ({positive/total*100:.2f}%)")
    filename = os.path.join(OUTPUT_DIR, f'claims_{claim_key}_monthly.parquet')
    df.to_parquet(filename, index=False)
print("\nAll files saved!")

## 7. Verify: No Same-Month Features

In [ ]:
sample = pd.read_parquet(os.path.join(OUTPUT_DIR, 'claims_fall_monthly.parquet'))
curr_cols = [c for c in sample.columns if '_curr_' in c]
print(f"Columns with '_curr_': {len(curr_cols)}")
if curr_cols:
    print("WARNING:", curr_cols)
else:
    print("CLEAN — no same-month features found.")

prev_cols = [c for c in sample.columns if '_prev' in c]
hist_cols = [c for c in sample.columns if '_hist_' in c]
print(f"\n_prev features: {len(prev_cols)}")
print(f"_hist_ features: {len(hist_cols)}")
print(f"Total columns: {len(sample.columns)}")